[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/16_planificacion_clasica/notebooks/aplicaciones/03_logistica.ipynb)

# Notebook 03 — Logística: independencia del dominio

En los notebooks anteriores implementamos un planificador basado en **STRIPS** y búsqueda hacia adelante (forward search) para resolver problemas del **mundo de bloques** (Blocks World). Ahora vamos a demostrar una de las propiedades más importantes de la planificación clásica: la **independencia del dominio**.

El algoritmo de búsqueda no cambia — lo único que cambia es la **definición del dominio**: las proposiciones y las acciones. El mismo planificador BFS que resolvió Blocks World puede resolver un problema de logística sin modificar una sola línea del motor de búsqueda.

Esto es exactamente lo que hace poderosa a la planificación clásica: separar el **mecanismo de razonamiento** (el planificador) de la **descripción del problema** (el dominio STRIPS).

---
## Parte 1: Infraestructura STRIPS

Redefinimos la infraestructura completa de STRIPS para que este notebook sea **autocontenido**. La representación es idéntica a la del Notebook 02:

- **Estado**: un `frozenset` de proposiciones (strings).
- **Acción**: una `namedtuple` con nombre, precondiciones, efectos positivos (add) y efectos negativos (delete).
- **Aplicabilidad**: una acción es aplicable si todas sus precondiciones están en el estado.
- **Transición**: aplicar una acción produce un nuevo estado = (estado - delete) ∪ add.

In [ ]:
from collections import namedtuple, deque

# STRIPS action representation
Action = namedtuple("Action", ["name", "pre", "add", "delete"])


def is_applicable(action, state):
    """Check whether all preconditions of the action hold in the state."""
    return action.pre.issubset(state)


def apply_action(action, state):
    """Return the successor state after applying the action.

    new_state = (state - delete_list) | add_list
    """
    return (state - action.delete) | action.add


print("Infraestructura STRIPS cargada.")

### Planificador BFS (forward search)

Usamos **búsqueda en amplitud (BFS)** sobre el espacio de estados. BFS garantiza encontrar un plan de longitud mínima (óptimo en número de pasos). El planificador es completamente **genérico** — recibe un estado inicial, un conjunto meta, y una función que genera las acciones aplicables.

In [ ]:
def bfs_planner(initial_state, goal, get_actions):
    """BFS forward planner for STRIPS problems.

    Parameters
    ----------
    initial_state : frozenset
        The initial state (set of true propositions).
    goal : frozenset
        The goal propositions (must all hold in a goal state).
    get_actions : callable(state) -> list[Action]
        Function that returns all applicable ground actions for a state.

    Returns
    -------
    plan : list[str] | None
        Sequence of action names, or None if no plan exists.
    explored : int
        Number of states explored during search.
    """
    if goal.issubset(initial_state):
        return [], 0

    frontier = deque([(initial_state, [])])  # (state, plan_so_far)
    visited = {initial_state}
    explored = 0

    while frontier:
        state, plan = frontier.popleft()
        explored += 1

        for action in get_actions(state):
            new_state = apply_action(action, state)
            if new_state in visited:
                continue
            new_plan = plan + [action.name]
            if goal.issubset(new_state):
                return new_plan, explored
            visited.add(new_state)
            frontier.append((new_state, new_plan))

    return None, explored  # no plan found


print("Planificador BFS cargado.")

---
## Parte 2: Dominio de logística

El dominio de **logística** modela el transporte de paquetes entre ubicaciones usando camiones. Es un dominio clásico en planificación, muy diferente del mundo de bloques.

### Proposiciones

| Proposición | Significado |
|-------------|-------------|
| `At(pkg, loc)` | El paquete `pkg` está en la ubicación `loc` |
| `In(pkg, truck)` | El paquete `pkg` está dentro del camión `truck` |
| `At(truck, loc)` | El camión `truck` está en la ubicación `loc` |

### Esquemas de acciones

| Acción | Precondiciones | Efectos (+) | Efectos (−) |
|--------|---------------|-------------|-------------|
| `Load(pkg, truck, loc)` | `At(pkg, loc)`, `At(truck, loc)` | `In(pkg, truck)` | `At(pkg, loc)` |
| `Unload(pkg, truck, loc)` | `In(pkg, truck)`, `At(truck, loc)` | `At(pkg, loc)` | `In(pkg, truck)` |
| `Drive(truck, from, to)` | `At(truck, from)` | `At(truck, to)` | `At(truck, from)` |

In [ ]:
def make_logistics_actions(packages, trucks, locations):
    """Generate all ground STRIPS actions for the logistics domain.

    Parameters
    ----------
    packages : list[str]
        Package identifiers, e.g. ["P1", "P2"].
    trucks : list[str]
        Truck identifiers, e.g. ["T1"].
    locations : list[str]
        Location identifiers, e.g. ["A", "B", "C"].

    Returns
    -------
    all_actions : list[Action]
        All ground (instantiated) actions for the domain.
    """
    actions = []

    # Load(pkg, truck, loc): load a package onto a truck at a location
    for pkg in packages:
        for truck in trucks:
            for loc in locations:
                actions.append(Action(
                    name=f"Load({pkg},{truck},{loc})",
                    pre=frozenset({f"At({pkg},{loc})", f"At({truck},{loc})"}),
                    add=frozenset({f"In({pkg},{truck})"}),
                    delete=frozenset({f"At({pkg},{loc})"}),
                ))

    # Unload(pkg, truck, loc): unload a package from a truck at a location
    for pkg in packages:
        for truck in trucks:
            for loc in locations:
                actions.append(Action(
                    name=f"Unload({pkg},{truck},{loc})",
                    pre=frozenset({f"In({pkg},{truck})", f"At({truck},{loc})"}),
                    add=frozenset({f"At({pkg},{loc})"}),
                    delete=frozenset({f"In({pkg},{truck})"}),
                ))

    # Drive(truck, from, to): move a truck between locations
    for truck in trucks:
        for loc_from in locations:
            for loc_to in locations:
                if loc_from == loc_to:
                    continue  # no-op drive
                actions.append(Action(
                    name=f"Drive({truck},{loc_from},{loc_to})",
                    pre=frozenset({f"At({truck},{loc_from})"}),
                    add=frozenset({f"At({truck},{loc_to})"}),
                    delete=frozenset({f"At({truck},{loc_from})"}),
                ))

    return actions


# Quick check: count the ground actions for a small domain
test_actions = make_logistics_actions(["P1", "P2"], ["T1"], ["A", "B", "C"])
print(f"Acciones instanciadas: {len(test_actions)}")
print(f"  Load:   {sum(1 for a in test_actions if a.name.startswith('Load'))}")
print(f"  Unload: {sum(1 for a in test_actions if a.name.startswith('Unload'))}")
print(f"  Drive:  {sum(1 for a in test_actions if a.name.startswith('Drive'))}")

---
## Parte 3: Problema pequeño — 2 paquetes, 1 camión, 3 ubicaciones

Definimos un problema concreto:

- **Paquetes**: P1, P2
- **Camión**: T1
- **Ubicaciones**: A, B, C

**Estado inicial**:
- P1 está en A
- P2 está en A
- T1 está en A

**Meta**:
- P1 debe estar en C
- P2 debe estar en B

In [ ]:
# Domain objects
packages_1 = ["P1", "P2"]
trucks_1 = ["T1"]
locations_1 = ["A", "B", "C"]

# Ground actions for this domain instance
all_actions_1 = make_logistics_actions(packages_1, trucks_1, locations_1)

# Initial state
init_1 = frozenset({
    "At(P1,A)",
    "At(P2,A)",
    "At(T1,A)",
})

# Goal
goal_1 = frozenset({
    "At(P1,C)",
    "At(P2,B)",
})


def get_applicable_1(state):
    """Return all applicable actions in the current state."""
    return [a for a in all_actions_1 if is_applicable(a, state)]


print("Estado inicial:", sorted(init_1))
print("Meta:", sorted(goal_1))

### Resolución con BFS

Ejecutamos el planificador BFS. Observa que usamos **exactamente el mismo algoritmo** que en Blocks World — solo cambiaron las acciones y proposiciones.

In [ ]:
plan_1, explored_1 = bfs_planner(init_1, goal_1, get_applicable_1)

if plan_1 is not None:
    print(f"Plan encontrado en {explored_1} estados explorados.")
    print(f"Longitud del plan: {len(plan_1)} acciones\n")
    print("Plan:")
    for i, step in enumerate(plan_1, 1):
        print(f"  {i}. {step}")
else:
    print("No se encontró plan.")

### Traza del plan

Veamos cómo evoluciona el estado paso a paso al ejecutar el plan encontrado.

In [ ]:
def trace_plan(initial_state, plan_names, all_actions):
    """Execute a plan and print the state after each action."""
    # Build a lookup from action name to Action object
    action_map = {a.name: a for a in all_actions}
    state = initial_state

    print("Estado inicial:")
    for prop in sorted(state):
        print(f"   {prop}")
    print()

    for i, name in enumerate(plan_names, 1):
        action = action_map[name]
        state = apply_action(action, state)
        print(f"Paso {i}: {name}")
        for prop in sorted(state):
            print(f"   {prop}")
        print()

    return state


final_state = trace_plan(init_1, plan_1, all_actions_1)

# Verify goal is satisfied
assert goal_1.issubset(final_state), "Error: la meta no se cumple."
print("La meta se cumple correctamente.")

---
## Parte 4: Problema más grande — 3 paquetes, 2 camiones, 4 ubicaciones

Ahora escalamos el problema para observar cómo crece el espacio de búsqueda. Con más objetos, el número de acciones instanciadas y estados posibles crece combinatoriamente.

- **Paquetes**: P1, P2, P3
- **Camiones**: T1, T2
- **Ubicaciones**: A, B, C, D

**Estado inicial**:
- P1 en A, P2 en B, P3 en A
- T1 en A, T2 en D

**Meta**:
- P1 en D, P2 en C, P3 en B

In [ ]:
# Larger domain
packages_2 = ["P1", "P2", "P3"]
trucks_2 = ["T1", "T2"]
locations_2 = ["A", "B", "C", "D"]

all_actions_2 = make_logistics_actions(packages_2, trucks_2, locations_2)

init_2 = frozenset({
    "At(P1,A)", "At(P2,B)", "At(P3,A)",
    "At(T1,A)", "At(T2,D)",
})

goal_2 = frozenset({
    "At(P1,D)", "At(P2,C)", "At(P3,B)",
})


def get_applicable_2(state):
    """Return all applicable actions for the larger domain."""
    return [a for a in all_actions_2 if is_applicable(a, state)]


print(f"Acciones instanciadas: {len(all_actions_2)}")
print(f"  Load:   {sum(1 for a in all_actions_2 if a.name.startswith('Load'))}")
print(f"  Unload: {sum(1 for a in all_actions_2 if a.name.startswith('Unload'))}")
print(f"  Drive:  {sum(1 for a in all_actions_2 if a.name.startswith('Drive'))}")
print(f"\nEstado inicial: {sorted(init_2)}")
print(f"Meta: {sorted(goal_2)}")

### Resolución del problema grande

Ejecutamos el mismo planificador BFS. Observa cuántos nodos se exploran en comparación con el problema pequeño.

In [ ]:
plan_2, explored_2 = bfs_planner(init_2, goal_2, get_applicable_2)

if plan_2 is not None:
    print(f"Plan encontrado en {explored_2} estados explorados.")
    print(f"Longitud del plan: {len(plan_2)} acciones\n")
    print("Plan:")
    for i, step in enumerate(plan_2, 1):
        print(f"  {i}. {step}")
else:
    print("No se encontró plan.")

print(f"\n--- Comparación ---")
print(f"Problema 1 (2 pkg, 1 truck, 3 loc): {explored_1:>6} estados explorados, plan de {len(plan_1)} pasos")
print(f"Problema 2 (3 pkg, 2 truck, 4 loc): {explored_2:>6} estados explorados, plan de {len(plan_2)} pasos")

---
## Parte 5: Discusión — Independencia del dominio

El resultado clave de este notebook es conceptualmente simple pero profundo:

> **El mismo algoritmo (BFS sobre STRIPS) resolvió tanto el mundo de bloques como logística sin ninguna modificación.**

Lo único que cambió fue la *definición del dominio*: las proposiciones y los esquemas de acciones. El planificador es **independiente del dominio** (*domain-independent*).

### Ventajas de la independencia del dominio

1. **Reutilización**: Un solo planificador sirve para cualquier dominio que se pueda expresar en STRIPS.
2. **Modularidad**: El experto del dominio define las acciones; el experto en IA mejora el algoritmo de búsqueda. Son esfuerzos ortogonales.
3. **Generalidad**: Nuevos problemas solo requieren una nueva especificación declarativa, no un nuevo programa.

### El costo de la generalidad

La independencia del dominio tiene un precio: el planificador no puede explotar la estructura particular de cada dominio. Un algoritmo diseñado específicamente para logística podría ser mucho más eficiente. Sin embargo, los planificadores modernos (como FF o FastDownward) usan **heurísticas derivadas automáticamente del dominio** para guiar la búsqueda — combinando independencia del dominio con eficiencia.

### Dominios adicionales que usan la misma representación

El formalismo STRIPS se ha aplicado exitosamente a cientos de dominios:
- Robótica (navegación, manipulación)
- Videojuegos (planificación de NPCs)
- Redes de computadoras (configuración automática)
- Manufactura (líneas de ensamblaje)
- Misiones espaciales (Mars rovers de la NASA)

Todos usan el mismo motor de planificación — solo cambia el archivo de dominio.